# 01 - Quickstart: matrix vs twin asymmetry in 30 lines

Synthetic Cu-Al-like grain population: load orientations + voxels, run K-medoids variant
assignment, compute Schmid factors, do a per-grain modified-WH fit, and write a master
inventory CSV.

Every step here is a one-liner against the `midas_defect` API; the real work is in
`notebooks/02_full_pipeline.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from midas_defect.types import CrystalPhase
from midas_defect.variants import assign_variants_kmeans, find_sigma3_partners
from midas_defect.schmid import schmid_factor_per_grain
from midas_defect.phases import FCC_SLIP_111_110
from midas_defect.line_profile import collect_per_grain_reflections, modified_wh_per_grain

# Pull in the shared synthetic fixture (60 grains, ~9600 voxels, 8 Cu hkls).
from tests.conftest import synthetic_cu_al_dataset  # type: ignore
import pytest
data = synthetic_cu_al_dataset.__wrapped__()  # invoke the bare fixture function

In [ ]:
# 1. K-medoids in disorientation metric (FCC -> SG 225). Recovers the planted Sigma3 split.
out = assign_variants_kmeans(data['OM'], n_variants=2, n_init=10, phase=CrystalPhase.FCC)
labels = out['labels']
print(f"counts:               {out['counts']}")
print(f"between-cluster diso: {out['disorientations'][0, 1]:.1f} deg")

In [ ]:
# 2. Sigma3 spatial matched pairs.
pairs = find_sigma3_partners(data['OM'], data['pos'], labels, k_NN=5, phase=CrystalPhase.FCC)
print(f"matched pairs: {pairs['pairs'].shape[0]}")
print(f"median misori: {np.median(pairs['pair_misori']):.1f} deg")

In [ ]:
# 3. Per-grain Schmid factor under uniaxial-z loading.
schmid = schmid_factor_per_grain(data['OM'], data['loading_axis'], FCC_SLIP_111_110)
plt.hist(schmid, bins=20); plt.xlabel('Schmid factor'); plt.ylabel('grain count'); plt.show()

In [ ]:
# 4. Per-grain modified-WH dislocation density.
entries = collect_per_grain_reflections(
    data['qs'], data['vals'], data['grain_of_voxel'], data['OM'], data['G_arr'],
    query_radius=0.20, min_voxels_per_refl=8,
)
wh = modified_wh_per_grain(entries, data['hkls'], burgers_length=data['burgers'])
print(f"median rho: {np.nanmedian(wh['rho_per_grain']):.2e} m^-2")
print(f"median q_U: {np.nanmedian(wh['q_U_per_grain']):.2f}")